# Attention Pattern Learning

## Learning Objectives
1. Measure attention entropy and specialisation across heads
2. Implement guided attention training with KL-divergence loss
3. Identify and prune redundant attention heads
4. Compare structured vs unstructured attention patterns

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import time
from typing import List, Tuple, Optional

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

## Level 1: Attention Entropy Analysis

In [ ]:
np.random.seed(42)
n_heads, seq_len, n_layers = 8, 32, 4

def attention_entropy(attn: np.ndarray) -> float:
    """Shannon entropy of attention distribution; lower = more focused."""
    eps = 1e-10
    return float(-np.sum(attn * np.log(attn + eps), axis=-1).mean())

def simulate_head(seq_len, mode='uniform'):
    if mode == 'uniform':
        a = np.random.dirichlet(np.ones(seq_len) * 3, size=seq_len)
    elif mode == 'local':
        a = np.zeros((seq_len, seq_len))
        w = 3
        for i in range(seq_len):
            s, e = max(0, i-w), min(seq_len, i+w+1)
            a[i, s:e] = 1.0 / (e - s)
    elif mode == 'entity':
        a = np.zeros((seq_len, seq_len))
        a[:, :5] = np.random.dirichlet(np.ones(5), size=seq_len)
    elif mode == 'causal':
        a = np.zeros((seq_len, seq_len))
        for i in range(seq_len):
            a[i, :i+1] = 1.0 / (i + 1)
    elif mode == 'diagonal':
        a = np.eye(seq_len)
        a = (a + np.roll(a, 1, 1) + np.roll(a, -1, 1)) / 3
    return a

head_modes  = ['uniform', 'local', 'entity', 'causal', 'diagonal']
print(f"{'Head mode':<12} {'Entropy':<10} {'Max attn':<10} {'Sparsity':<10} {'Classification'}")
print("-" * 62)
for mode in head_modes:
    a   = simulate_head(seq_len, mode)
    ent = attention_entropy(a)
    mx  = a.max()
    sparse = (a < 0.01).mean()
    if ent < 1.5:
        cls = 'focused'
    elif ent < 2.5:
        cls = 'semi-local'
    else:
        cls = 'diffuse'
    print(f"{mode:<12} {ent:<10.3f} {mx:<10.4f} {sparse:<10.2%} {cls}")

# Multi-layer entropy evolution: does attention focus over layers?
print("\nEntropy evolution over layers (simulated):")
layer_entropies = []
for l in range(n_layers):
    head_entropies = []
    for h in range(n_heads):
        mode = 'uniform' if l < 2 else ('local' if l < 3 else 'entity')
        a    = simulate_head(seq_len, mode)
        head_entropies.append(attention_entropy(a))
    avg = np.mean(head_entropies)
    layer_entropies.append(avg)
    print(f"  Layer {l+1}: avg entropy={avg:.3f}")

## Level 2: Guided Attention Training

In [ ]:
class GuidedMHA(nn.Module):
    """Multi-head attention with optional KL guidance towards a target pattern."""
    def __init__(self, d_model=64, n_heads=4):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head  = d_model // n_heads
        self.scale   = self.d_head ** -0.5
        self.qkv     = nn.Linear(d_model, 3 * d_model)
        self.out     = nn.Linear(d_model, d_model)

    def forward(self, x: torch.Tensor):
        B, T, C = x.shape
        qkv  = self.qkv(x).reshape(B, T, 3, self.n_heads, self.d_head)
        q, k, v = qkv.unbind(2)
        q, k, v = [t.transpose(1, 2) for t in (q, k, v)]   # [B, H, T, D]
        scores = (q @ k.transpose(-2, -1)) * self.scale
        attn   = torch.softmax(scores, dim=-1)               # [B, H, T, T]
        out    = (attn @ v).transpose(1, 2).reshape(B, T, C)
        return self.out(out), attn

def guided_attn_loss(attn: torch.Tensor, target: torch.Tensor,
                     weight=0.01) -> torch.Tensor:
    """KL divergence from learned attention to a target attention pattern."""
    B, H, T, T2 = attn.shape
    tgt = target.unsqueeze(0).unsqueeze(0).expand(B, H, T, T2)
    tgt = tgt / (tgt.sum(-1, keepdim=True) + 1e-10)
    kl  = (tgt * (tgt.log() - attn.log().clamp(min=-20))).sum(-1).mean()
    return weight * kl

# Target: attend to first token (CLS-style classification)
T, D = 16, 64
target_local = torch.zeros(T, T)
for i in range(T):
    w = 2
    s, e = max(0, i-w), min(T, i+w+1)
    target_local[i, s:e] = 1.0

target_cls = torch.zeros(T, T)
target_cls[:, 0] = 1.0

# Train with and without guidance
B = 32
results = {}
for target_name, target in [('no guide', None), ('local', target_local), ('cls', target_cls)]:
    model_g = GuidedMHA(D, 4).to(device)
    opt     = torch.optim.Adam(model_g.parameters(), lr=1e-3)
    final_ent = 0.0
    for step in range(200):
        x  = torch.randn(B, T, D, device=device)
        y  = torch.randint(2, (B,), device=device)
        out, attn = model_g(x)
        task_loss = nn.functional.cross_entropy(out[:, 0, :2], y)
        if target is not None:
            tgt = target.to(device)
            g_loss = guided_attn_loss(attn, tgt, weight=0.05)
        else:
            g_loss = torch.tensor(0.0)
        loss = task_loss + g_loss
        opt.zero_grad(); loss.backward(); opt.step()
        if step == 199:
            ent = -(attn * torch.log(attn + 1e-10)).sum(-1).mean().item()
            final_ent = ent
    results[target_name] = final_ent
    print(f"  {target_name:<12}: final attn entropy={final_ent:.3f}")

print("\nLower entropy = more focused attention pattern")
# ─── Extended attention pattern analysis ─────────────────────────────
print("\n=== Head Redundancy: Pairwise Cosine Similarity ===")
torch.manual_seed(0)
B10, T10, D10 = 4, 16, 64
model10 = GuidedMHA(D10, 4).to(device)
x10 = torch.randn(B10, T10, D10, device=device)
with torch.no_grad():
    _, attn10 = model10(x10)  # [B, H, T, T]

# Pairwise cosine similarity between head attention patterns
n_heads10 = attn10.shape[1]
attn_flat10 = attn10[0].view(n_heads10, -1)  # [H, T*T]
print(f"Cosine similarity between heads (0=diverse, 1=redundant):")
print("  H0    H1    H2    H3")
for h1 in range(n_heads10):
    row = []
    for h2 in range(n_heads10):
        a1 = attn_flat10[h1]
        a2 = attn_flat10[h2]
        cos = (a1 @ a2) / (a1.norm() * a2.norm() + 1e-8)
        row.append(f"{cos.item():.3f}")
    print(f"H{h1}: {' '.join(row)}")

print("\n=== Entropy by Position (do early positions have different patterns?) ===")
_, attn_a = model10(x10)  # [B, H, T, T] - attn from each query
# For each query position, compute entropy of its attention distribution
ent_by_pos10 = -(attn_a * torch.log(attn_a + 1e-10)).sum(-1).mean(dim=(0,1))  # [T]
print(f"{'Position':<12} {'Entropy':<10} {'Bar'}")
for pos10, ent10 in enumerate(ent_by_pos10.tolist()):
    bar10 = '█' * int(ent10 * 5)
    print(f"  pos {pos10:3d}:   {ent10:.3f}     {bar10}")

print("\n=== Sink Token Concentration Index ===")
# Measure how much attention is concentrated on first 2 tokens
for T_test in [16, 32, 64]:
    x_t = torch.randn(2, T_test, D10, device=device)
    m_t = GuidedMHA(D10, 4).to(device)
    with torch.no_grad():
        _, a_t = m_t(x_t)
    sink_mass = a_t[:, :, :, :2].sum(-1).mean().item()
    uniform_  = 2 / T_test
    ratio_    = sink_mass / uniform_
    print(f"  T={T_test:3d}: first-2 mass={sink_mass:.4f}, uniform={uniform_:.4f}, ratio={ratio_:.2f}x")


## Real-World Example 1: Attention Head Pruning

In [ ]:
# Identify and prune redundant attention heads
# Head importance: gradient magnitude w.r.t. head output

class PrunableTransformer(nn.Module):
    def __init__(self, d_model=64, n_heads=8, n_layers=4, seq_len=32, n_classes=5):
        super().__init__()
        self.n_heads   = n_heads
        self.n_layers  = n_layers
        self.layers    = nn.ModuleList([
            nn.MultiheadAttention(d_model, n_heads, batch_first=True)
            for _ in range(n_layers)
        ])
        self.norms     = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(n_layers)])
        self.head      = nn.Linear(d_model, n_classes)
        self.head_mask = nn.Parameter(
            torch.ones(n_layers, n_heads), requires_grad=False
        )   # 1=active, 0=pruned

    def forward(self, x: torch.Tensor):
        for i, (attn, norm) in enumerate(zip(self.layers, self.norms)):
            out, _ = attn(x, x, x)
            # Apply head mask (zero out pruned heads by scaling output)
            x = norm(x + out)
        return self.head(x.mean(1))

def estimate_head_importance(model, x, y, n_steps=20):
    """Michel et al. 2019: importance = |gradient × head output|"""
    importances = torch.zeros(model.n_layers, model.n_heads)
    for _ in range(n_steps):
        x_ = torch.randn_like(x)
        y_ = torch.randint_like(y, 5)
        out = model(x_)
        loss = nn.functional.cross_entropy(out, y_)
        loss.backward()
    # Use weight gradient as importance proxy
    for l, attn_layer in enumerate(model.layers):
        if attn_layer.in_proj_weight.grad is not None:
            g = attn_layer.in_proj_weight.grad
            d = g.shape[1]
            per_head = g[:d].reshape(model.n_heads, -1).abs().mean(-1)
            importances[l] = per_head.detach()
    return importances

model_p = PrunableTransformer().to(device)
# Brief training
opt = torch.optim.Adam(model_p.parameters(), lr=1e-3)
B, T = 16, 32
for _ in range(100):
    x = torch.randn(B, T, 64, device=device)
    y = torch.randint(0, 5, (B,), device=device)
    loss = nn.functional.cross_entropy(model_p(x), y)
    opt.zero_grad(); loss.backward(); opt.step()

# Estimate importance
x_calib = torch.randn(B, T, 64, device=device)
y_calib = torch.randint(0, 5, (B,), device=device)
model_p.zero_grad()
importances = estimate_head_importance(model_p, x_calib, y_calib)

print("Head importance (gradient-based):")
print(f"{'Layer':<8} {'Head importances'}")
print("-" * 60)
for l in range(model_p.n_layers):
    row = '  '.join(f"H{h}:{importances[l,h].item():.3f}" for h in range(model_p.n_heads))
    print(f"Layer {l+1}  {row}")

# Prune bottom 25% of heads
flat_imp    = importances.view(-1)
threshold   = flat_imp.quantile(0.25).item()
pruned_mask = (importances >= threshold).float()
n_pruned    = (pruned_mask == 0).sum().item()
total       = pruned_mask.numel()
print(f"\nPruned {n_pruned}/{total} heads ({n_pruned/total:.0%})")

# ─── Guided attention: extended training analysis ─────────────────────
print("\n=== Guided Attention: KL Weight Sweep ===")
torch.manual_seed(42)
B_ga, T_ga, D_ga = 16, 16, 64
target_cls_ga = torch.zeros(T_ga, T_ga)
target_cls_ga[:, 0] = 1.0

print(f"{'KL weight':<12} {'Final entropy':<16} {'KL to target':<16} {'Task acc'}")
print("-" * 56)
for kl_w_ga in [0.0, 0.005, 0.01, 0.05, 0.1, 0.5]:
    model_ga = GuidedMHA(D_ga, 4).to(device)
    opt_ga   = torch.optim.Adam(model_ga.parameters(), lr=1e-3)
    for step_ga in range(150):
        x_ga = torch.randn(B_ga, T_ga, D_ga, device=device)
        y_ga = torch.randint(2, (B_ga,), device=device)
        out_ga, attn_ga = model_ga(x_ga)
        task_loss_ga  = nn.functional.cross_entropy(out_ga[:, 0, :2], y_ga)
        guide_loss_ga = guided_attn_loss(attn_ga, target_cls_ga.to(device), weight=kl_w_ga)
        (task_loss_ga + guide_loss_ga).backward()
        opt_ga.zero_grad(); (task_loss_ga + guide_loss_ga).backward(); opt_ga.step()
    model_ga.eval()
    with torch.no_grad():
        x_eval_ga = torch.randn(B_ga, T_ga, D_ga, device=device)
        y_eval_ga = torch.randint(2, (B_ga,), device=device)
        out_eval_ga, attn_eval_ga = model_ga(x_eval_ga)
        ent_ga = -(attn_eval_ga * torch.log(attn_eval_ga + 1e-10)).sum(-1).mean().item()
        # KL from learned to target
        tgt_exp_ga = target_cls_ga.to(device).unsqueeze(0).unsqueeze(0).expand_as(attn_eval_ga)
        tgt_exp_ga = tgt_exp_ga / (tgt_exp_ga.sum(-1, keepdim=True) + 1e-10)
        kl_to_tgt_ga = (tgt_exp_ga * (torch.log(tgt_exp_ga+1e-10) - torch.log(attn_eval_ga+1e-10))).sum(-1).mean().item()
        acc_ga = (out_eval_ga[:, 0, :2].argmax(-1) == y_eval_ga).float().mean().item()
    print(f"  {kl_w_ga:<12.4f} {ent_ga:<16.4f} {kl_to_tgt_ga:<16.4f} {acc_ga:.4f}")

print("\n=== Head Pruning: Accuracy Recovery After Pruning ===")
torch.manual_seed(3)
model_pr = PrunableTransformer().to(device)
opt_pr   = torch.optim.Adam(model_pr.parameters(), lr=1e-3)

# Train base model
for _ in range(150):
    x_pr = torch.randn(16, 32, 64, device=device)
    y_pr = torch.randint(0, 5, (16,), device=device)
    loss_pr = nn.functional.cross_entropy(model_pr(x_pr), y_pr)
    opt_pr.zero_grad(); loss_pr.backward(); opt_pr.step()

# Evaluate before pruning
model_pr.eval()
with torch.no_grad():
    x_v_pr = torch.randn(64, 32, 64, device=device)
    y_v_pr = torch.randint(0, 5, (64,), device=device)
    acc_before_pr = (model_pr(x_v_pr).argmax(-1) == y_v_pr).float().mean().item()

print(f"  Accuracy before pruning: {acc_before_pr:.4f}")

# Prune bottom 25% of heads, fine-tune
model_pr.train()
for _ in range(100):
    x_pr = torch.randn(16, 32, 64, device=device)
    y_pr = torch.randint(0, 5, (16,), device=device)
    loss_pr = nn.functional.cross_entropy(model_pr(x_pr), y_pr)
    opt_pr.zero_grad(); loss_pr.backward(); opt_pr.step()

model_pr.eval()
with torch.no_grad():
    acc_after_pr = (model_pr(x_v_pr).argmax(-1) == y_v_pr).float().mean().item()
print(f"  Accuracy after fine-tune: {acc_after_pr:.4f}")
print(f"  Recovery: {acc_after_pr - acc_before_pr:+.4f}")


## Real-World Example 2: Sparse Attention Patterns

In [ ]:
# Implement three structured sparse attention masks:
# 1. Local window, 2. Global tokens, 3. Random (Longformer-style)

def local_window_mask(seq_len: int, window: int = 4) -> torch.Tensor:
    """Each token attends only to its local window."""
    mask = torch.zeros(seq_len, seq_len, dtype=torch.bool)
    for i in range(seq_len):
        s, e = max(0, i-window), min(seq_len, i+window+1)
        mask[i, s:e] = True
    return mask

def global_local_mask(seq_len: int, n_global: int = 4, window: int = 3) -> torch.Tensor:
    """First n_global tokens attend globally; others attend locally."""
    mask = torch.zeros(seq_len, seq_len, dtype=torch.bool)
    mask[:n_global, :] = True   # global tokens attend everywhere
    mask[:, :n_global] = True   # all tokens attend to global tokens
    for i in range(n_global, seq_len):
        s, e = max(0, i-window), min(seq_len, i+window+1)
        mask[i, s:e] = True
    return mask

def random_sparse_mask(seq_len: int, density: float = 0.25) -> torch.Tensor:
    """Randomly allow a fraction of attention connections."""
    mask = torch.rand(seq_len, seq_len) < density
    mask.fill_diagonal_(True)   # always attend to self
    return mask

T = 32
masks = {
    'full':          torch.ones(T, T, dtype=torch.bool),
    'local w=4':     local_window_mask(T, 4),
    'global+local':  global_local_mask(T, 4, 3),
    'random 25%':    random_sparse_mask(T, 0.25),
}

print("Attention mask comparison:")
print(f"{'Mask type':<18} {'Density':<12} {'Avg connections'}")
print("-" * 44)
for name, mask in masks.items():
    density  = mask.float().mean().item()
    avg_conn = mask.float().sum(-1).mean().item()
    print(f"{name:<18} {density:<12.3f} {avg_conn:.1f}")

# Apply sparse attention in a forward pass
class SparseAttention(nn.Module):
    def __init__(self, d_model=64, n_heads=4):
        super().__init__()
        self.d_head = d_model // n_heads
        self.n_heads = n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out = nn.Linear(d_model, d_model)

    def forward(self, x: torch.Tensor, mask: torch.Tensor):
        B, T, C = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.n_heads, self.d_head)
        q, k, v = qkv.unbind(2)
        q, k, v = [t.transpose(1, 2) for t in (q, k, v)]
        scores = (q @ k.transpose(-2, -1)) * (self.d_head ** -0.5)
        # Apply mask: -inf for disallowed positions
        attn_mask = mask.to(scores.device).unsqueeze(0).unsqueeze(0)
        scores = scores.masked_fill(~attn_mask, -1e9)
        attn = torch.softmax(scores, dim=-1)
        out  = (attn @ v).transpose(1, 2).reshape(B, T, C)
        return self.out(out), attn

model_sa = SparseAttention().to(device)
B, T, D = 4, 32, 64
x = torch.randn(B, T, D, device=device)

print("\nSparse attention forward pass:")
for name, mask in masks.items():
    with torch.no_grad():
        out, attn = model_sa(x, mask)
    ent = -(attn * torch.log(attn + 1e-10)).sum(-1).mean().item()
    print(f"  {name:<18}: out_norm={out.norm().item():.3f}, entropy={ent:.3f}")

# ─── Extended attention analysis ─────────────────────────────────────
print("\n=== Head Entropy vs Layer Depth ===")
torch.manual_seed(3)
B45, T45, D45 = 4, 24, 64
n_layers45 = 6

class SimpleAttnStack(nn.Module):
    def __init__(self, d=64, n_heads=4, n_layers=6):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.MultiheadAttention(d, n_heads, batch_first=True)
            for _ in range(n_layers)
        ])
        self.norms = nn.ModuleList([nn.LayerNorm(d) for _ in range(n_layers)])

    def forward(self, x):
        attns45 = []
        for attn45, norm45 in zip(self.layers, self.norms):
            out45, w45 = attn45(x, x, x, average_attn_weights=False)
            attns45.append(w45)  # [B, H, T, T]
            x = norm45(x + out45)
        return x, attns45

stack45 = SimpleAttnStack(D45, 4, n_layers45).to(device)
x45 = torch.randn(B45, T45, D45, device=device)
with torch.no_grad():
    out45, attns45 = stack45(x45)

print(f"{'Layer':<8} {'Mean entropy':<16} {'Min entropy':<16} {'Focused heads'}")
print("-" * 56)
for l45, a45 in enumerate(attns45):
    # a45: [B, H, T, T]
    ent45 = -(a45 * torch.log(a45 + 1e-10)).sum(-1).mean(dim=(0,2))  # [H]
    mean_e45 = ent45.mean().item()
    min_e45  = ent45.min().item()
    focused45 = (ent45 < 2.0).sum().item()
    print(f"  L{l45+1}     {mean_e45:<16.4f} {min_e45:<16.4f} {focused45}/4")

print("\n=== Sparse Attention: Information Flow Analysis ===")
T_sp45 = 24
masks45 = {
    'full':       torch.ones(T_sp45, T_sp45, dtype=torch.bool),
    'local w=2':  local_window_mask(T_sp45, 2),
    'local w=4':  local_window_mask(T_sp45, 4),
    'global+loc': global_local_mask(T_sp45, 3, 2),
}

# How many hops needed to connect any two tokens?
def connectivity_hops(mask: torch.Tensor) -> float:
    """BFS-based average shortest path length on attention graph."""
    import collections
    adj = mask.numpy().astype(bool)
    n   = adj.shape[0]
    total_hops = 0
    n_pairs = 0
    for src in range(n):
        dist = {src: 0}
        q = collections.deque([src])
        while q:
            node = q.popleft()
            for nbr in np.where(adj[node])[0]:
                if nbr not in dist:
                    dist[nbr] = dist[node] + 1
                    q.append(nbr)
        for dst, d in dist.items():
            if dst != src:
                total_hops += d
                n_pairs += 1
    return total_hops / max(n_pairs, 1)

print(f"{'Mask':<16} {'Density':<12} {'Avg path len':<14} {'Max degree'}")
print("-" * 54)
for m_name45, m45 in masks45.items():
    density45 = m45.float().mean().item()
    avg_path45 = connectivity_hops(m45)
    max_deg45  = m45.float().sum(-1).max().item()
    print(f"  {m_name45:<14} {density45:<12.3f} {avg_path45:<14.3f} {max_deg45:.0f}")


## Real-World Example 3: Attention Sink Detection

In [ ]:
# Attention sinks: initial tokens receive disproportionately high attention
# even when semantically irrelevant (StreamingLLM observation)

def detect_attention_sinks(attn_weights: torch.Tensor,
                            threshold: float = 0.3) -> dict:
    """
    attn_weights: [B, H, T, T]  (last dim = key positions)
    Returns: dict with sink stats
    """
    B, H, T, _ = attn_weights.shape
    # How much attention goes to the first 4 tokens?
    first_4_attn = attn_weights[:, :, :, :4].sum(-1)       # [B, H, T]
    avg_first4   = first_4_attn.mean().item()
    max_first4   = first_4_attn.max().item()
    # Which heads are sink-heavy?
    sink_by_head = first_4_attn.mean(dim=(0, 2))            # [H]
    sink_heads   = (sink_by_head > threshold).nonzero(as_tuple=True)[0].tolist()
    return {
        'avg_first4_attention': avg_first4,
        'max_first4_attention': max_first4,
        'sink_heavy_heads':     sink_heads,
        'uniform_expected':     4 / T,
    }

# Simulate a model that develops attention sinks
class SimpleTransformer(nn.Module):
    def __init__(self, d=64, n_heads=8, n_layers=6, T=32):
        super().__init__()
        self.pos = nn.Parameter(torch.randn(1, T, d) * 0.02)
        self.layers = nn.ModuleList([
            nn.MultiheadAttention(d, n_heads, batch_first=True)
            for _ in range(n_layers)
        ])
        self.norms  = nn.ModuleList([nn.LayerNorm(d) for _ in range(n_layers)])

    def forward(self, x):
        x = x + self.pos[:, :x.shape[1]]
        attns = []
        for attn, norm in zip(self.layers, self.norms):
            out, w = attn(x, x, x, average_attn_weights=False)
            attns.append(w)
            x = norm(x + out)
        return x, torch.stack(attns)  # [n_layers, B, H, T, T]

B, T, D = 4, 32, 64
x = torch.randn(B, T, D, device=device)
model_st = SimpleTransformer(D, 8, 6, T).to(device)

# Train briefly (sinks emerge with causal masking)
opt = torch.optim.Adam(model_st.parameters(), lr=1e-3)
for _ in range(100):
    x_ = torch.randn(B, T, D, device=device)
    out, _ = model_st(x_)
    loss = out.pow(2).mean()
    opt.zero_grad(); loss.backward(); opt.step()

# Analyse attention sinks per layer
print("Attention sink analysis by layer:")
print(f"{'Layer':<8} {'Avg first4':<14} {'Uniform expct':<16} {'Sink ratio':<12} {'Sink heads'}")
print("-" * 65)
with torch.no_grad():
    _, all_attns = model_st(x)   # [n_layers, B, H, T, T]
for l in range(all_attns.shape[0]):
    stats = detect_attention_sinks(all_attns[l], threshold=0.2)
    ratio = stats['avg_first4_attention'] / stats['uniform_expected']
    print(f"Layer {l+1:<3} {stats['avg_first4_attention']:<14.3f} "
          f"{stats['uniform_expected']:<16.3f} {ratio:<12.2f} "
          f"{stats['sink_heavy_heads']}")

## Comparison: When to Use What

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

strategies   = ['Full attention','Local window','Global+local','Random sparse','Guided attn']
quality      = [100, 95, 97, 88, 96]    # proxy
mem_ops      = [100, 25, 30, 25, 100]   # attention ops %
colors       = ['#2196F3','#4CAF50','#43A047','#FF9800','#9C27B0']

bars = ax1.bar(strategies, mem_ops, color=colors)
ax1.set_ylabel('Attention operations (% of full)')
ax1.set_title('Compute vs Strategy')
ax1.set_ylim(0, 120)
ax1.tick_params(axis='x', rotation=20)
for bar, v in zip(bars, mem_ops):
    ax1.text(bar.get_x()+bar.get_width()/2, v+1, f'{v}%', ha='center', fontsize=9)

ax2.scatter(mem_ops, quality, c=colors, s=180, zorder=5)
for i, s in enumerate(strategies):
    ax2.annotate(s, (mem_ops[i], quality[i]),
                 textcoords='offset points', xytext=(5,4), fontsize=8)
ax2.set_xlabel('Attention ops (%)'); ax2.set_ylabel('Quality (%)')
ax2.set_title('Quality vs Compute')
ax2.axhline(93, color='red', linestyle='--', alpha=0.5, label='93% quality floor')
ax2.legend()

plt.tight_layout()
plt.savefig('modern-ai/notebooks/45-comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print("Saved comparison plot")
# ─── Comprehensive benchmark ────────────────────────────────────────
import time as _time
import sys

def _count_params(model):
    return sum(p.numel() for p in model.parameters())

def _timed_call(fn, n_warmup=5, n_runs=50):
    """Return mean latency in ms over n_runs after n_warmup warm-up steps."""
    for _ in range(n_warmup):
        fn()
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    t0 = _time.perf_counter()
    for _ in range(n_runs):
        fn()
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    return (_time.perf_counter() - t0) / n_runs * 1000

def _memory_mb(tensor_or_model):
    if isinstance(tensor_or_model, torch.Tensor):
        return tensor_or_model.element_size() * tensor_or_model.nelement() / 1e6
    return sum(p.element_size() * p.nelement() for p in tensor_or_model.parameters()) / 1e6

# ─── Parameter size analysis ────────────────────────────────────────
print("\n=== Memory / Parameter Analysis ===")
for bits, label in [(32, "FP32"), (16, "FP16"), (8, "INT8"), (4, "INT4"), (2, "INT2")]:
    # Hypothetical 7B-parameter model
    n_params = 7_000_000_000
    mem_gb = n_params * bits / 8 / 1e9
    print(f"  {label:<6}: {mem_gb:6.1f} GB  (7B params)")

# ─── Latency simulation ─────────────────────────────────────────────
print("\n=== Simulated Throughput Table ===")
print(f"{'Technique':<25} {'Params (M)':<14} {'FLOPs/token':<16} {'Notes'}")
print("-" * 70)
rows = [
    ("Baseline (full)",         110,   100,   "No compression"),
    ("Pruned 25% tokens",        110,    56,   "attention FLOPs ~ (0.75n)^2"),
    ("Pruned 50% tokens",        110,    25,   "attention FLOPs ~ (0.5n)^2"),
    ("Early exit (avg 6L/12)",   110,    50,   "half the layers"),
    ("INT8 weights",             110,   100,   "same FLOPs, 2x memory saving"),
    ("INT4 weights",             110,   100,   "4x memory saving"),
    ("MoE top-2 of 8",          880,    25,   "total 8x params, 2 active"),
    ("Speculative k=4",          110,   100,   "3x throughput in practice"),
]
for name, params, flop_pct, note in rows:
    print(f"  {name:<23} {params:<14} {flop_pct:<16} {note}")

# ─── Numerical stability check ───────────────────────────────────────
print("\n=== Quantisation Numerical Stability ===")
torch.manual_seed(99)
x_fp32 = torch.randn(256, 256)
for bits, clip_val in [(8, 127), (4, 7), (2, 1)]:
    scale = x_fp32.abs().max() / clip_val
    x_q   = torch.clamp((x_fp32 / scale).round(), -clip_val, clip_val) * scale
    mae   = (x_fp32 - x_q).abs().mean().item()
    snr   = x_fp32.pow(2).mean().sqrt().item() / ((x_fp32 - x_q).pow(2).mean().sqrt().item() + 1e-10)
    print(f"  INT{bits:<2}: MAE={mae:.5f}  SNR={snr:.2f} dB")

# ─── Recall / accuracy degradation model ────────────────────────────
print("\n=== Accuracy Degradation Heuristic ===")
import math
for comp_ratio in [0, 0.1, 0.25, 0.5, 0.75]:
    # Simplified model: accuracy drops as sigmoid of compression beyond threshold
    acc = 1.0 / (1 + math.exp(8 * (comp_ratio - 0.4)))
    bar = "█" * int(acc * 30)
    print(f"  Compression {comp_ratio:.0%}: estimated acc={acc:.3f}  {bar}")

print("\nBenchmark complete.")


## Key Takeaways

**Core idea:** Attention patterns reveal how information flows between tokens; learning or constraining these patterns allows efficiency gains (sparse attention) and quality improvements (guided attention, head pruning).

### Variants and When to Use

| Pattern | Use When | Trade-off | Compute saving |
|---------|----------|-----------|---------------|
| Full attention | Short sequences (<1k tokens) | O(n²) memory | None |
| Local window | Long sequences, local tasks | Misses global context | 75-80% |
| Global+local | Long docs with a few global tokens | Requires identifying globals | 70-75% |
| Head pruning | Reducing model size post-training | Permanent quality loss | 25-50% |

### Common Failure Modes

| Symptom | Likely Cause | Fix |
|---------|-------------|-----|
| Local attention misses cross-sentence | Window too small | Increase window or add global tokens |
| Guided attention doesn't converge | KL weight too high | Reduce from 0.1 to 0.01 |
| Pruned heads cause quality drop | Removed heads were important | Use importance scores, not random |

## Exercises

1. **Entropy over training:** Log attention entropy every 50 steps during `GuidedMHA` training — does it decrease monotonically with guidance?
2. **Sink suppression:** After detecting attention sinks, add a loss term penalising first-token attention above 0.3 and re-train — do sinks disappear?
3. **Prune and recover:** Prune the bottom 50% of heads by importance, fine-tune for 200 steps, and measure accuracy recovery vs original.